# 🐙 Gemini Autonomous Agent

**Multi-module edition** — `router → planner → executor → reflect` + hybrid skill store.

Runs 100% on the **Google AI Studio free tier**.

---
**Get your free API key →** https://aistudio.google.com/apikey

## Step 1 — Clone the repo

In [ ]:
!git clone https://github.com/yashmgupta/cuddly-octopus.git
%cd cuddly-octopus/gemini-agent
!ls

## Step 2 — Install dependencies

In [ ]:
!pip install -q google-genai googlesearch-python python-dotenv rich
print('✅ All dependencies installed.')

## Step 3 — Set your Gemini API key

In [ ]:
import os

# ← Paste your free key from https://aistudio.google.com/apikey
os.environ['GEMINI_API_KEY'] = 'PASTE_YOUR_KEY_HERE'

# Optional: change model (default is gemini-2.5-flash)
# os.environ['MODEL'] = 'gemini-2.5-flash'

assert os.environ['GEMINI_API_KEY'] != 'PASTE_YOUR_KEY_HERE', '❌ Please replace PASTE_YOUR_KEY_HERE with your actual API key!'
print('✅ API key set.')

## Step 4 — Import modules & initialise the agent

In [ ]:
import sys
sys.path.insert(0, '.')

from agent import memory
from agent import skills as skill_store
from main import execute_task

# Initialise SQLite databases
memory.init_db()
skill_store.init_skills_db()

print('✅ Agent initialised — databases ready.')

## Step 5 — Run the agent

### Test A — existing built-in skill (weather)

In [ ]:
# Should match the built-in 'fetch_weather' skill directly
await execute_task("What's the weather like?")

### Test B — trigger NEW skill (web search)

In [ ]:
# No skill exists yet → planner + ReAct loop + skill generated & saved
await execute_task("Search for the latest news about AI agents")

### Test C — reuse previously saved skill

In [ ]:
# After Test B the skill is saved; this should MATCH it immediately
await execute_task("Find recent articles about large language models")

### Test D — write & read a file

In [ ]:
await execute_task("Write a short poem about octopuses and save it to poem.txt")

## Step 6 — Inspect what was learned

In [ ]:
import json
from pathlib import Path

skills_file = Path('data/skills_database.json')
if skills_file.exists():
    saved = json.loads(skills_file.read_text())
    print(f'📦 {len(saved)} learned skill(s) in JSON store:\n')
    for name, data in saved.items():
        print(f'  🔧 {name}: {data["description"]}')
else:
    print('No skills saved yet — run Tests B/C/D first.')

In [ ]:
import sqlite3

db_path = 'data/agent.db'
if Path(db_path).exists():
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    print('📊 SQLite skill stats:\n')
    rows = conn.execute('SELECT name, trigger, success_count, fail_count, updated_at FROM skills').fetchall()
    for r in rows:
        print(f"  {r['name']:30s}  ✅ {r['success_count']}  ❌ {r['fail_count']}  | trigger: {r['trigger'][:50]}")

    print('\n🧠 Recent memories:\n')
    rows = conn.execute('SELECT content, importance, created_at FROM memories ORDER BY id DESC LIMIT 5').fetchall()
    for r in rows:
        print(f"  [{r['importance']}] {r['content'][:80]}...  ({r['created_at'][:19]})")
    conn.close()
else:
    print('No SQLite DB yet — run some tasks first.')

## Step 7 — Custom prompt (try your own!)

Edit the string below and run the cell.

In [ ]:
# ✏️ Change this to anything you want
your_prompt = "Search for Python tutorials for beginners"

await execute_task(your_prompt)